# OpenMC Cross Section Helpers — Quickstart Examples

This notebook demonstrates a few minimal examples for each public function of `openmc-xs-helpers`:

- `available_library_reactions()`
- `available_reactions()`
- `peak_cross_section()`
- `cross_section_at_energy()`
- `peak_xs_table()`
- `find_xs_table()`
- `plot_xs()`


## Setup (if you don’t already have OpenMC)

This notebook requires **OpenMC + a neutron data library** (a valid `cross_sections.xml` and the referenced HDF5 files). If you don’t have OpenMC installed locally (common on Windows), the easiest way is to use the **`fusion-energy/neutronics-workshop`** Docker image.

### 1) Install prerequisites
- Install **Docker Desktop**
- Make sure Docker is running

### 2) Clone this repository
Run these commands in a terminal:

    git clone https://github.com/lindsaydowne/openmc-xs-helpers.git
    cd openmc-xs-helpers

### 3) Start the container and mount this repo
From the repo root (where `pyproject.toml` lives), run:

    docker run --rm -it \
      -p 8888:8888 \
      -v "$(pwd):/work" \
      ghcr.io/fusion-energy/neutronics-workshop

### 4) Install this package inside the container (editable install)
Inside the container, run:

    cd /work
    python -m pip install -e .

### 5) Confirm OpenMC + nuclear data are visible
Still inside the container, run:

    python -c "import openmc, os; print('openmc', openmc.__version__); print('cross_sections:', openmc.config.get('cross_sections')); print('ENV:', os.environ.get('OPENMC_CROSS_SECTIONS'))"

You should see a valid `cross_sections.xml` path.

### 6) Launch JupyterLab
Inside the container, run:

    jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root

Open the URL printed in the terminal (it includes a token), then open: `examples/01_quickstart.ipynb`

---

### Notes
- If port `8888` is already in use, change the **host** port only (left side), e.g. `-p 8880:8888`.
- This notebook assumes `openmc.config["cross_sections"]` is set (either by the container or via `OPENMC_CROSS_SECTIONS`).


In [ ]:
import os
import openmc
import openmc_xs_helpers as xsh

# Locations of the 
print("openmc.__version__:", getattr(openmc, "__version__", "unknown"))
print("openmc.config['cross_sections']:", openmc.config.get("cross_sections", None))
print("ENV OPENMC_CROSS_SECTIONS:", os.environ.get("OPENMC_CROSS_SECTIONS", None))


## 0) Create a material

We will make UO2 fuel as a material to be able to demonstrate the functionality of some of our public functions.

In [ ]:
# Change enrichment to see how the results change.
enrichment = 0.0495

materials = openmc.Materials()

UO2_fuel = openmc.Material(name="UO2_fuel")
UO2_fuel.temperature = 900.0
UO2_fuel.add_nuclide("U235",percent=1.0/3.0*enrichment,percent_type="ao");
UO2_fuel.add_nuclide("U238",percent=1.0/3.0*(1-enrichment),percent_type="ao");
UO2_fuel.add_element("O", 2.0)
UO2_fuel.set_density("g/cm3", 10.4)

materials.append(UO2_fuel)
materials.export_to_xml()

print("materials.xml written")

We can now start demonstrating how each function works. 

## 1) `available_reactions()`
Many of the functions can take the reaction name eg. `"(n,gamma)"` or `"elastic"` or their associated MT number `(MT=2)`, `(MT =102)`.

There are many MT numbers however not all nuclides have every one and so we have built a function to show which reactions MTs are present in the library for the target(s). This works for single nuclides  (eg. `"Gd-157"`), elements (`"Gd"`) or materials (`UO2_fuel`). 

It is useful to discover what you can request by MT (or by reaction string, if supported). 
The default is to exclude the scattering and derived reactions.

In [ ]:
rx_gd = xsh.available_reactions("Gd") # use strings for elements or nuclides eg. ("Gd-157")
print(rx_gd)                 # pretty string: "(n,elastic) (MT=2), ..."

In [ ]:
rx_w = xsh.available_reactions(
    targets = UO2_fuel,          # do not use a string for an openmc material. 
    temperature = "900K",        # accepted for call consistency; MT availability is temperature-independent
    #xs_xml_path = "/opt/openmc_data/cross_sections.xml" # Only specify this if you have more than one cross section library.
    # max_items = None,          # lists all reactions and mts
    exclude_scattering = False,  # scattering are MT=53-91 and are (n,n') with residual in the 3-40th excited state & continuum.
    exclude_derived = False,     # very specific from MT=219-999 that are not useful for initial understanding. 
)
print(rx_w)

## 2) `peak_cross_section()`

It is sometimes useful to be able to find the peak of a cross section over an energy range. This function only works for one nuclide + one MT (or reaction) because it is used within the `peak_xs_table` function which will will be able to list out the peak cross sections for numerous nuclides or reactions. We will see this later on. 

The default energy range is from 0.0253 eV (thermal) to 16 MeV (max fusion relevant energies) but this can be changed within the function. 

In [ ]:
# Example: capture (n,gamma) is typically MT=102
pk_Gd157 = xsh.peak_cross_section("Gd157", "elastic")
pk_Gd157

# It calculates the peak as `xs_b_peak` and the energy of the peak in both eV and MeV `E_eV_peak`, `E_MeV_peak`.
# notice the other default settings.

In [ ]:
# Example: elastic peak for uranium isotope
pk_w = xsh.peak_cross_section(
    nuclide = "U-235",                 # only works for nuclides and not full elements or materials.
    reaction_or_mt = "(n,gamma)",
    #temperature = "1200K",
    energy_min_eV = 0.0253,  # default fusion value = 0.0253 eV, only works needs to be in eV rather than keV or MeV.
    energy_max_eV = 1e6 ,  # default fusion value = 16 MeV, only works needs to be in eV rather than keV or MeV.
    #xs_xml_path = "/opt/openmc_data/cross_sections.xml",   # Only specify this if you have more than one cross section library.
)

pk_w



## 3) `cross_section_at_energy()`

Similarly we have also created a function that returns the cross section at a specific neutron energy.
Energy can be given as:
- numeric (assumed eV)
- strings like `"1 eV"`, `"14 MeV"`, `"100 keV"`


In [ ]:
# Gd157 capture at thermal-ish energy
xsh.cross_section_at_energy("Gd157", 102, temperature="294K", neutron_energy="0.0253 eV")


In [ ]:
# W184 elastic at 14 MeV
xsh.cross_section_at_energy(
    nuclide = "Li-6",         # only works for nuclides and not full elements or materials.
    reaction_or_mt = "(n,t)",
    temperature = "900K",
    neutron_energy = "5 keV", # works as a string with eV, keV or MeV
    ##xs_xml_path = "/opt/openmc_data/cross_sections.xml",
)

## 4) `peak_xs_table()`

We mostly want to be able to view the cross sections of multiple nuclides side by side. This function creates a ranked table of peak cross sections:
- If `reactions_or_mts=None`, it uses "major reactions" (auto) and ranks by peak.
- For element targets (e.g. `"Gd"`), it uses natural isotopic abundances for atom fractions.


In [ ]:
# Element target (natural abundances) chooses the top 15 
xsh.peak_xs_table("Gd", temperature="294K", top_n=15)


In [ ]:
# Explicit MT list
xsh.peak_xs_table("Gd", reactions_or_mts=[2, 102, 16], temperature="294K")

In [ ]:
# Also works on materials.
xsh.peak_xs_table(
    targets = UO2_fuel,
    # reactions_or_mts = ["(n,Xn)", "(n,fission)"],
    # temperature = "294K",
    # energy_min_eV =  0.0253,
    # energy_max_eV = 2e6, 
    # xs_xml_path = "/opt/openmc_data/cross_sections.xml",
    top_n = 20,  # None => print all
    scale_by_atom_fraction = True, # scales cross sections by atom fraction before finding the peak. 
                                   # Change to True and see how the XS change.
) 

## 5) `find_xs_table()`

Same table idea, but ranking is done at a specified energy (`neutron_energy=...`).


In [ ]:
# At 1 eV (should show only non-zero rows if you added the filter)
xsh.find_xs_table("Gd", temperature="294K", neutron_energy="1 eV", top_n=20)


In [ ]:
# At 2 MeV: fast spectrum example
xsh.find_xs_table(
    targets = UO2_fuel,
    reactions_or_mts = "(n,fission)",     # will pull out all (n,fission) cross sections  
    # temperature = None,                 # will default to "294K"
    neutron_energy =  "2 MeV",
    # energy_min_eV = 1, 
    # energy_max_eV = 2e7,
    # xs_xml_path = "/opt/openmc_data/cross_sections.xml",
    # top_n = None,  
    scale_by_atom_fraction = False,       # change to True and see how the XS change. 
)

## 6) `plot_xs()`

A lot of the time a comparison of cross sections through plotting them is necessary to understand the behaviour of a material. This function creates an interactive Plotly plot with:
- mode buttons (which toggle between microscopic cross sections, microscopic cross sections × atom-fractions, and combined macroscopic cross section for the material.)
- X axis toggle (which toggle between linear energy scales in MeV or logarithmic energy scales in eV)
- Y axis toggle (log vs linear )

Tip: Start with an element like `"Gd"` or a single nuclide like `"Li6"` to keep traces manageable.


In [ ]:
# Explicit reactions (keeps the plot simple)
xsh.plot_xs("Li-6", reactions_or_mts=[2, 102], temperature="294K", top_n=10, verbose=False)

# Note the cross section buttons don't show for simple nuclides as the three modes are identical.

In [ ]:
# It works for multiple elements or nuclides in a list. 
# We have selected top_n = 5 to show only the top 5 dominant reactions (for each mode)
xsh.plot_xs(
    targets = ["Gd", "W-186"],
    #reactions_or_mts = "(n,fission)",  # None => auto top 10 major reactions
    #temperature = None,  # default 294K
    #energy_min_eV = 0.0253,
    #energy_max_eV = 2e6,
    top_n = 5,        # default = 10
    #xs_xml_path = "/opt/openmc_data/cross_sections.xml",
    #scale_by_atom_fraction = True,                # This 
    verbose = True,
)

# Start playing around with this one. Plotly allows you to zoom in and move around really easily. 
# Click on a legend entry to remove / add from the plot and double click to isolate/return all. 

In [ ]:
# And finally, it is just a simple 1 line of code to plot the dominant cross sections of a material. 

xsh.plot_xs(UO2_fuel)

## 7) `available_library_reactions()`

If you would like to view all possible reactions and MTs that are present in the neutron data library then run this function. This is really only useful to discover what you can request by MT (or by reaction string, if supported) and will take a long time to process. 


In [ ]:
print(xsh.available_library_reactions(max_nuclides = 100))
# This may take a couple of minutes.

In [ ]:
# If we want to see the full list of reactions and MTs: 
print(
    xsh.available_library_reactions(
        #xs_xml_path="/opt/openmc_data/cross_sections.xml",
        #max_nuclides=100,            # this will make it quicker
        exclude_scattering=False,     # don't remove the scattering cross sections
        exclude_derived=False,        # don't remove the derived cross sections
        as_available_reactions=True,  # True returns as reaction - MT pairs, False just returns list of MTs.
        verbose=True,
    )
)

# this may tke a while while as it is searching through all of the nuclides.